# 数理変換タクティクス AI トレーニング
## ハイブリッドEmbedding版（数学特徴 + 学習特徴 + 継続学習 + リソース対応）

In [ ]:
!pip install tensorflowjs scikit-learn -q
print('完了！')

In [ ]:
import numpy as np
import tensorflow as tf
import tensorflowjs as tfjs
import random, os, math
from sklearn.model_selection import train_test_split

MAX_HAND    = 20
EMBED_DIM   = 16
NUM_ACTIONS = 7
MAX_CARD    = 150
PRIMES      = [2, 3, 5, 7, 11, 13]
MATH_DIM    = len(PRIMES) + 3
SCALAR_DIM  = 10
DIM_NAMES   = ['2の指数','3の指数','5の指数','7の指数','11の指数','13の指数','大素数フラグ','約数の個数','桁の和']

print(f'TensorFlow: {tf.__version__}')

## Google Drive 連携

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR  = '/content/drive/MyDrive/math_tactics_ai'
SAVE_PATH = os.path.join(SAVE_DIR, 'ai_best.keras')
DATA_PATH = os.path.join(SAVE_DIR, 'train_data.npz')
os.makedirs(SAVE_DIR, exist_ok=True)

print('モデル保存先: ' + SAVE_PATH)
print('データ保存先: ' + DATA_PATH)
if os.path.exists(SAVE_PATH): print('保存済みモデルあり')
if os.path.exists(DATA_PATH): print('保存済みデータあり')

## 数学特徴テーブル

In [ ]:
def card_math_features(n):
    if n == 0: return [0.0] * MATH_DIM
    orig = n; factors = []
    for p in PRIMES:
        count = 0
        while n % p == 0: n //= p; count += 1
        factors.append(min(count, 4) / 4.0)
    has_large = 1.0 if n > 1 else 0.0
    ndiv = sum(1 for d in range(1, orig+1) if orig % d == 0)
    ds   = sum(int(d) for d in str(orig))
    return factors + [has_large, min(ndiv,20)/20.0, min(ds,30)/30.0]

FACTOR_TABLE = np.array([card_math_features(i) for i in range(MAX_CARD+1)], dtype=np.float32)
print('テーブル shape: ' + str(FACTOR_TABLE.shape))

## ゲームシミュレーター（リソース対応）

In [ ]:
def gcd(a,b):
    while b: a,b=b,a%b
    return a
def digit_sum(n): return sum(int(d) for d in str(n))

class MathCardGame:
    def __init__(self, config=None):
        self.config = config or {'initHandCount':5,'initMaxNum':10,'drawCount':2,'drawMaxNum':20,'handLimitNum':MAX_CARD,'winScore':100,'maxTurns':10,'resourceInitial':10,'resourceLogBase':2}
    def reset(self):
        cfg=self.config
        self.hands={'P1':[random.randint(1,cfg['initMaxNum']) for _ in range(cfg['initHandCount'])],'P2':[random.randint(1,cfg['initMaxNum']) for _ in range(cfg['initHandCount'])]}
        self.scores={'P1':0,'P2':0}; self.turn_count=1; self.current_player='P1'
        self.done=False; self.winner=None
        ri = cfg.get('resourceInitial', None)
        self.resources = {'P1': ri, 'P2': ri}
    def opponent(self,pid): return 'P2' if pid=='P1' else 'P1'
    def _res_delta(self, hb, ha):
        ri=self.config.get('resourceInitial', None)
        if ri is None: return 0.0
        diff = sum(ha) - sum(hb)
        if diff == 0: return 0.0
        base = max(self.config.get('resourceLogBase', 2), 1.001)
        return (1 if diff>0 else -1) * math.log(abs(diff)+1) / math.log(base)
    def _can_afford(self, pid, hb, ha):
        ri=self.config.get('resourceInitial', None)
        if ri is None: return True
        return (self.resources.get(pid, ri) + self._res_delta(hb, ha)) >= 0
    def _use_resource(self, pid, hb, ha):
        ri=self.config.get('resourceInitial', None)
        if ri is None: return
        self.resources[pid] = self.resources.get(pid, ri) + self._res_delta(hb, ha)
    def _res_ratio(self, pid):
        ri=self.config.get('resourceInitial', None)
        if ri is None or ri <= 0: return 1.0
        return min(self.resources.get(pid, ri) / ri, 2.0)
    def get_inputs(self,pid):
        cfg=self.config; my=self.hands[pid]; enm=self.hands[self.opponent(pid)]
        pad=lambda arr:(arr[:MAX_HAND]+[0]*MAX_HAND)[:MAX_HAND]
        return pad(my),pad(enm),\
               [self.scores[pid]/100.0, self.scores[self.opponent(pid)]/100.0,
                self.turn_count/10.0, len(my)/MAX_HAND, len(enm)/MAX_HAND,
                cfg['winScore']/200.0, cfg['maxTurns']/20.0, cfg['initHandCount']/10.0,
                self._res_ratio(pid), self._res_ratio(self.opponent(pid))]
    def best_attack(self,pid):
        enm=self.hands[self.opponent(pid)]; best,bg=None,0
        for i,a in enumerate(self.hands[pid]):
            if a==1: continue
            g=sum(n for n in enm if n%a==0)
            if g>bg: bg=g; best=(i,a,g)
        return best
    def best_add(self,pid):
        h,lim=self.hands[pid],self.config['handLimitNum']; best,bv=None,-1
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j: continue
                r=h[i]+h[j]
                if r<=lim and r>bv: bv=r; best=(i,j,r)
        return best
    def best_sub(self,pid):
        h=self.hands[pid]; best,bv=None,0
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j: continue
                r=h[i]-h[j]
                if r>0 and r>bv: bv=r; best=(i,j,r)
        return best
    def best_divprod(self,pid):
        h,lim=self.hands[pid],self.config['handLimitNum']; best,bv=None,-1
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j or h[j]==0: continue
                r=(h[i]//h[j])*(h[i]%h[j])
                if r>0 and r<=lim and r>bv: bv=r; best=(i,j,r)
        return best
    def best_dsd(self,pid):
        h,lim=self.hands[pid],self.config['handLimitNum']; best,bv=None,-1
        for i,num in enumerate(h):
            ds=digit_sum(num)
            if ds==0: continue
            q,r=divmod(num,ds)
            if q<=lim and q>bv: bv=q; best=(i,q,r)
        return best
    def best_gm(self,pid):
        h,lim=self.hands[pid],self.config['handLimitNum']; best,bv=None,-1
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j: continue
                g=gcd(h[i],h[j])
                if g<=1: continue
                r=h[i]*g
                if r<=lim and r>bv: bv=r; best=(i,j,r)
        return best
    def get_optimal_action(self,pid):
        h=self.hands[pid]
        atk=self.best_attack(pid); add=self.best_add(pid); sub=self.best_sub(pid)
        dp=self.best_divprod(pid); dsd=self.best_dsd(pid); gm=self.best_gm(pid)
        ms=self.scores[pid]; win=self.config['winScore']
        def atk_ok(): return bool(atk) and self._can_afford(pid,h,[c for k,c in enumerate(h) if k!=atk[0]])
        def sub_ok():
            if not sub: return False
            i,j,r=sub; return self._can_afford(pid,h,[c for k,c in enumerate(h) if k!=i and k!=j]+[r])
        def dp_ok():
            if not dp: return False
            i,j,r=dp; return self._can_afford(pid,h,[c for k,c in enumerate(h) if k!=i and k!=j]+[r])
        def dsd_ok():
            if not dsd: return False
            idx,q,r=dsd; nh=[c for k,c in enumerate(h) if k!=idx]+[q]+([] if r==0 else [r])
            return self._can_afford(pid,h,nh)
        if atk and self.turn_count>1 and ms+atk[2]>=win and atk_ok(): return 1
        if gm  and gm[2]>=30:  return 6
        if atk and self.turn_count>1 and atk[2]>=10 and atk_ok(): return 1
        if add and add[2]>=20: return 2
        if dp  and dp[2]>=15 and dp_ok(): return 4
        if atk and self.turn_count>1 and atk_ok(): return 1
        if gm  and gm[2]>=10: return 6
        if dsd and dsd[1]>=5 and dsd_ok(): return 5
        if add: return 2
        if dp and dp_ok(): return 4
        if sub and sub_ok(): return 3
        return 0
    def execute_action(self,pid,action):
        h=self.hands[pid]; eid=self.opponent(pid)
        if action==1:
            atk=self.best_attack(pid)
            if atk:
                i,v,gain=atk; b=h[:]; a=[c for k,c in enumerate(h) if k!=i]
                if self._can_afford(pid,b,a):
                    self._use_resource(pid,b,a); self.hands[eid]=[n for n in self.hands[eid] if n%v!=0]
                    h.pop(i); self.scores[pid]+=gain
                    if self.scores[pid]>=self.config['winScore']: self.done=True; self.winner=pid
        elif action==2:
            a=self.best_add(pid)
            if a:
                i,j,r=a; b=h[:]; h=[c for k,c in enumerate(h) if k!=i and k!=j]+[r]
                self._use_resource(pid,b,h)
        elif action==3:
            s=self.best_sub(pid)
            if s:
                i,j,r=s; b=h[:]; nh=[c for k,c in enumerate(h) if k!=i and k!=j]+[r]
                if self._can_afford(pid,b,nh): self._use_resource(pid,b,nh); h=nh
        elif action==4:
            d=self.best_divprod(pid)
            if d:
                i,j,r=d; b=h[:]; nh=[c for k,c in enumerate(h) if k!=i and k!=j]+[r]
                if self._can_afford(pid,b,nh): self._use_resource(pid,b,nh); h=nh
        elif action==5:
            d=self.best_dsd(pid)
            if d:
                idx,q,r=d; b=h[:]; nh=[c for k,c in enumerate(h) if k!=idx]+[q]+([] if r==0 else [r])
                if self._can_afford(pid,b,nh): self._use_resource(pid,b,nh); h=nh
        elif action==6:
            g=self.best_gm(pid)
            if g:
                i,j,r=g; b=h[:]; h=[c for k,c in enumerate(h) if k!=i and k!=j]+[r]
                self._use_resource(pid,b,h)
        self.hands[pid]=h; self.current_player=eid
        if self.current_player=='P1':
            self.turn_count+=1
            if self.turn_count>self.config['maxTurns']: self.done=True; self.winner=max(self.scores,key=lambda p:self.scores[p])
print('シミュレーター準備完了！')

## トレーニングデータの読み込み または 生成

In [ ]:
if os.path.exists(DATA_PATH):
    print('保存済みデータを読み込みます: ' + DATA_PATH)
    d=np.load(DATA_PATH)
    my_hands_data=d['my_hands']; enemy_hands_data=d['enemy_hands']
    scalars_data=d['scalars'];   y_data=d['y']
    print('読み込み完了！ %d 件' % len(y_data))
else:
    print('データ生成を開始します...')
    NUM_EPISODES=30000
    RULE_CONFIGS=[
        {'initHandCount':3, 'initMaxNum':10,'drawCount':2,'drawMaxNum':20,'handLimitNum':MAX_CARD,'winScore':50, 'maxTurns':8, 'resourceInitial':5,   'resourceLogBase':2},
        {'initHandCount':5, 'initMaxNum':10,'drawCount':2,'drawMaxNum':20,'handLimitNum':MAX_CARD,'winScore':100,'maxTurns':10,'resourceInitial':10,  'resourceLogBase':2},
        {'initHandCount':5, 'initMaxNum':15,'drawCount':2,'drawMaxNum':30,'handLimitNum':MAX_CARD,'winScore':150,'maxTurns':15,'resourceInitial':15,  'resourceLogBase':2},
        {'initHandCount':7, 'initMaxNum':20,'drawCount':3,'drawMaxNum':50,'handLimitNum':MAX_CARD,'winScore':200,'maxTurns':20,'resourceInitial':20,  'resourceLogBase':3},
        {'initHandCount':10,'initMaxNum':10,'drawCount':1,'drawMaxNum':10,'handLimitNum':MAX_CARD,'winScore':100,'maxTurns':10,'resourceInitial':10,  'resourceLogBase':2},
        {'initHandCount':5, 'initMaxNum':10,'drawCount':2,'drawMaxNum':20,'handLimitNum':MAX_CARD,'winScore':100,'maxTurns':10,'resourceInitial':None,'resourceLogBase':2}
    ]
    my_hands_data,enemy_hands_data,scalars_data,y_data=[],[],[],[]
    for ep in range(NUM_EPISODES):
        cfg=random.choice(RULE_CONFIGS); game=MathCardGame(cfg); game.reset()
        while not game.done:
            pid=game.current_player
            my_h,enm_h,sc=game.get_inputs(pid)
            action=game.get_optimal_action(pid)
            my_hands_data.append(my_h); enemy_hands_data.append(enm_h)
            scalars_data.append(sc);    y_data.append(action)
            game.execute_action(pid,action)
        if (ep+1)%10000==0: print('%d/%d データ数: %d' % (ep+1,NUM_EPISODES,len(y_data)))
    my_hands_data=np.array(my_hands_data,dtype=np.int32)
    enemy_hands_data=np.array(enemy_hands_data,dtype=np.int32)
    scalars_data=np.array(scalars_data,dtype=np.float32)
    y_data=np.array(y_data,dtype=np.int32)
    print('生成完了！ 総データ数: %d' % len(y_data))
    np.savez_compressed(DATA_PATH,my_hands=my_hands_data,enemy_hands=enemy_hands_data,scalars=scalars_data,y=y_data)
    print('データを保存しました: ' + DATA_PATH)

## モデルの作成 または 読み込み

In [ ]:
from tensorflow.keras.utils import to_categorical

idx=np.arange(len(y_data))
train_idx,val_idx=train_test_split(idx,test_size=0.1,random_state=42)

def make_dataset(indices,batch_size=256,shuffle=True):
    ds=tf.data.Dataset.from_tensor_slices((
        {'my_hand':my_hands_data[indices],'enemy_hand':enemy_hands_data[indices],'scalars':scalars_data[indices]},
        to_categorical(y_data[indices],NUM_ACTIONS)))
    if shuffle: ds=ds.shuffle(10000)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds=make_dataset(train_idx); val_ds=make_dataset(val_idx,shuffle=False)

if os.path.exists(SAVE_PATH):
    print('保存済みモデルを読み込み中...')
    model=tf.keras.models.load_model(SAVE_PATH)
    model.get_layer('math_emb').set_weights([FACTOR_TABLE])
    print('読み込み完了！続きから学習します。')
else:
    print('新規モデルを作成します...')
    my_hand_in   =tf.keras.Input(shape=(MAX_HAND,),  dtype='int32',  name='my_hand')
    enemy_hand_in=tf.keras.Input(shape=(MAX_HAND,),  dtype='int32',  name='enemy_hand')
    scalar_in    =tf.keras.Input(shape=(SCALAR_DIM,),dtype='float32',name='scalars')
    math_emb   =tf.keras.layers.Embedding(MAX_CARD+1,MATH_DIM,weights=[FACTOR_TABLE],trainable=False,mask_zero=True,name='math_emb')
    learned_emb=tf.keras.layers.Embedding(MAX_CARD+1,EMBED_DIM,mask_zero=True,name='learned_emb')
    def encode_hand(hand_input):
        combined=tf.keras.layers.Concatenate(axis=-1)([math_emb(hand_input),learned_emb(hand_input)])
        return tf.keras.layers.GlobalMaxPooling1D()(combined)
    x=tf.keras.layers.Concatenate(name='combined')([encode_hand(my_hand_in),encode_hand(enemy_hand_in),scalar_in])
    x=tf.keras.layers.Dense(128,activation='relu')(x)
    x=tf.keras.layers.Dropout(0.3)(x)
    x=tf.keras.layers.Dense(64, activation='relu')(x)
    x=tf.keras.layers.Dropout(0.2)(x)
    out=tf.keras.layers.Dense(NUM_ACTIONS,activation='softmax',name='action')(x)
    model=tf.keras.Model(inputs=[my_hand_in,enemy_hand_in,scalar_in],outputs=out)
    print('新規モデル作成完了！')

model.compile(optimizer=tf.keras.optimizers.Adam(0.001),loss='categorical_crossentropy',metrics=['accuracy'])
total=model.count_params(); trainable=sum(tf.size(w).numpy() for w in model.trainable_weights)
print('総パラメータ: %d  学習: %d  固定: %d' % (total,trainable,total-trainable))

## 学習

In [ ]:
callbacks=[
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy',patience=6,restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=3),
    tf.keras.callbacks.ModelCheckpoint(filepath=SAVE_PATH,monitor='val_accuracy',save_best_only=True,verbose=1)
]
history=model.fit(train_ds,validation_data=val_ds,epochs=60,callbacks=callbacks,verbose=1)
best_val=max(history.history['val_accuracy'])
print('最高検証精度: %.1f%%' % (best_val*100))

## 精度グラフ

In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,key,title in zip(axes,['accuracy','loss'],['精度','損失']):
    ax.plot(history.history[key],label='訓練')
    ax.plot(history.history['val_'+key],label='検証')
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()

## 数学特徴の重要度

In [ ]:
import matplotlib.pyplot as plt
base_loss,base_acc=model.evaluate(val_ds,verbose=0)
print('ベースライン精度: %.2f%%' % (base_acc*100))
importances=[]
for d in range(MATH_DIM):
    shuffled=FACTOR_TABLE.copy()
    shuffled[:,d]=np.random.permutation(shuffled[:,d])
    model.get_layer('math_emb').set_weights([shuffled])
    _,acc=model.evaluate(val_ds,verbose=0)
    drop=base_acc-acc; importances.append(drop)
    model.get_layer('math_emb').set_weights([FACTOR_TABLE])
    print('次元%2d [%s]: %+.2f%%' % (d+1,DIM_NAMES[d],drop*100))
plt.figure(figsize=(10,4))
colors=['crimson' if v>0.02 else 'steelblue' for v in importances]
plt.barh(DIM_NAMES,[v*100 for v in importances],color=colors)
plt.axvline(0,color='black',linewidth=0.8)
plt.xlabel('精度の低下 (%)')
plt.title('数学特徴の重要度')
plt.tight_layout(); plt.show()
print('最も重要: ' + DIM_NAMES[int(np.argmax(importances))])

## TensorFlow.js エクスポート

In [ ]:
import shutil
OUT='tfjs_model'
tfjs.converters.save_keras_model(model,OUT)
print('エクスポート完了！')
for f in os.listdir(OUT): print('  %s (%.1f KB)' % (f,os.path.getsize(os.path.join(OUT,f))/1024))
shutil.make_archive('tfjs_model','zip','tfjs_model')
from google.colab import files
files.download('tfjs_model.zip')
print('ダウンロード開始！')

## 動作テスト

In [ ]:
names=['パス','攻撃','足し算','引き算','商x余','桁和分裂','GCD掛け']
std_cfg={'initHandCount':5,'initMaxNum':10,'drawCount':2,'drawMaxNum':20,
         'handLimitNum':MAX_CARD,'winScore':100,'maxTurns':10,'resourceInitial':10,'resourceLogBase':2}
tests=[
    {'p1':[6,3,10,7,2], 'p2':[12,18,5,9,4],    'desc':'攻撃チャンスあり'},
    {'p1':[5,8,3,2,7],  'p2':[1,1,1,1,1],       'desc':'合成が有利'},
    {'p1':[6,9,4,3,2],  'p2':[8,10,6,4,3],      'desc':'GCD有効'},
    {'p1':[36,12,24,48],'p2':[7,11,13,17,19,23],'desc':'相手が全員素数'}
]
for t in tests:
    g=MathCardGame(std_cfg); g.reset()
    g.hands['P1']=t['p1']; g.hands['P2']=t['p2']
    g.turn_count=3; g.scores={'P1':20,'P2':30}; g.resources={'P1':8.0,'P2':8.0}
    my_h,enm_h,sc=g.get_inputs('P1')
    pred=model.predict({'my_hand':np.array([my_h]),'enemy_hand':np.array([enm_h]),'scalars':np.array([sc])},verbose=0)[0]
    chosen=int(np.argmax(pred)); optimal=g.get_optimal_action('P1')
    mark='OK' if chosen==optimal else 'NG'
    print('[%s] %s  AI:%s  最適:%s' % (mark,t['desc'],names[chosen],names[optimal]))